# Title

# Summary

# Introduction

# Exploratory Data Analysis

## Part 1: Data Ingestion & Data Wrangling  
Since raw data has no NA values, this section begins with loading the red wine and white wine datasets from `data/raw/` directory separately, adding labels indicating the wine type and combining them into a single unified dataset for further analysis.  


In [1]:
import pandas as pd

In [2]:
path_red = "data/raw/winequality-red.csv"
path_white = "data/raw/winequality-white.csv"
df_red = pd.read_csv(path_red, sep=";")
df_white = pd.read_csv(path_white, sep=";")

Below adding two new columns:  
 - `wine_type`: `red` for red wine, `white` for white wine. This column is used only for EDA and will not be included as a feature for classification.
 - `target`: `0` for red wine, `white` for white wine, which is the variable to be predicted.

In [3]:
df_red["wine_type"] = "red"
df_white["wine_type"] = "white"
df_red["target"] = 0
df_white["target"] = 1

Then merging the datasets and removing `quality` column:
 - two datasets are concatenated row-wise into one dataframe called `df_merge`
 - `quality` score is highly informative and could lead to data leakage if used as a feature when predicting wine type, therefore, we drop it from `df_merge`.

In [4]:
df_merge = pd.concat([df_red, df_white], axis=0).reset_index(drop=True)
df_merge = df_merge.drop(columns=["quality"])
df_merge.columns = df_merge.columns.str.replace(" ", "_")
df_merge

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,wine_type,target
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,red,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,red,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,red,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,red,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,red,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,white,1
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,white,1
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,white,1
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,white,1


In [5]:
df_merge["wine_type"].value_counts()

wine_type
white    4898
red      1599
Name: count, dtype: int64

The final combined dataset contains:
 - all numeric chemical features
 - `wine_type` (categorical, only for EDA visualization), with 4,898 white wine samples and 1599 red wine samples.
 - `target` (binary variable for classification)  
  
Now this dataframe is not imbalanced and clean, it is ready for EDA and modeling.

## Part 2: Exploratory Data Analysis  
This section summarizes the data structure and observing important statistical properties that related to the red vs white wine classification task.
### 1. Data Structure

In [6]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df_merge, test_size=0.2, random_state=123)
df_info = train_df.info()
df_info

<class 'pandas.core.frame.DataFrame'>
Index: 5197 entries, 6452 to 3582
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed_acidity         5197 non-null   float64
 1   volatile_acidity      5197 non-null   float64
 2   citric_acid           5197 non-null   float64
 3   residual_sugar        5197 non-null   float64
 4   chlorides             5197 non-null   float64
 5   free_sulfur_dioxide   5197 non-null   float64
 6   total_sulfur_dioxide  5197 non-null   float64
 7   density               5197 non-null   float64
 8   pH                    5197 non-null   float64
 9   sulphates             5197 non-null   float64
 10  alcohol               5197 non-null   float64
 11  wine_type             5197 non-null   object 
 12  target                5197 non-null   int64  
dtypes: float64(11), int64(1), object(1)
memory usage: 568.4+ KB


The `.info()` output shows:  
 - No missing values
 - 11 numerical features

Confirms that the dataset is clean and ready for analysis.

In [7]:
df_summary = train_df.describe()
df_summary

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,target
count,5197.000000,5197.000000,5197.000000,5197.000000,5197.000000,5197.00000,5197.000000,5197.000000,5197.000000,5197.000000,5197.000000,5197.000000
mean,7.215519,0.338230,0.320073,5.451472,0.056028,30.73629,116.182990,0.994690,3.217897,0.530069,10.493335,0.756206
std,1.298654,0.164058,0.146037,4.769121,0.035179,17.90455,56.898123,0.003019,0.160178,0.149313,1.200092,0.429411
min,3.800000,0.080000,0.000000,0.600000,0.012000,1.00000,6.000000,0.987110,2.720000,0.220000,8.000000,0.000000
25%,6.400000,0.230000,0.250000,1.800000,0.038000,17.00000,78.000000,0.992300,3.110000,0.430000,9.500000,1.000000
50%,7.000000,0.290000,0.310000,3.000000,0.047000,29.00000,118.000000,0.994890,3.210000,0.500000,10.300000,1.000000
75%,7.700000,0.400000,0.390000,8.100000,0.065000,42.00000,156.000000,0.996990,3.320000,0.600000,11.300000,1.000000
max,15.900000,1.580000,1.660000,65.800000,0.611000,289.00000,440.000000,1.038980,4.010000,2.000000,14.900000,1.000000


Using `.describe()` to check summary statistics for all numerical variables.  
Key observations:
 - **Residual sugar**, **free sulfur dioxide**, and **total sulfur dioxide** have wide ranges and right-skewed distributions with extreme outliers.
 - **pH**, **alcohol**, **density**, **chlorides** and **critic acid** have small standard deviations, showing relatively tight distributions.

These statistics provide insight into which variables may separate wine types well.

### 2. Univariate Distributions
Overall histogram patterns:
 - Many features (e.g. **residual sugar**, **chlorides**, **sulphates**) show right-skewness, which means most wines fall in lower ranges with a few extreme values, whereas **alchohol** is slightly right-skewed.
 - The distribution of **dnesity** and **pH** are similar to Normal distribution.


In [8]:
import altair as alt
alt.data_transformers.enable("vegafusion")

numeric_cols = train_df.select_dtypes('number').columns.tolist()

plots = []
for col in numeric_cols:
    plot = alt.Chart(train_df).mark_bar().encode(
        alt.X(
            col, type="quantitative",
            bin=alt.Bin(maxbins=40),
            title=f"{col} (binned)"
        ),
        alt.Y("count()",title="Count").stack(False),
    ).properties(
        width=140,
        height=110
    )
    plots.append(plot)

hist_plot = alt.concat(*plots, columns=3).properties(
    title="Univariate Distributions of Numeric Features"
).configure_title(fontSize=20)
hist_plot

alt.ConcatChart(...)

### 3. Pairwise Correlations
Overall correlation patterns:
 - **Density** is strongly positively correlated with **residual sugar**, and **free sulfur dioxide** is strongly correlated with **total sulfur dioxide**.
 - **Alcohol** is negatively correlated with **density**, consistent with wine chemistry.
 - **pH** is negatively correlated with **fixed acidity**, showing the expected inverse relationship.


In [9]:
corr = train_df[numeric_cols].corr().reset_index().melt("index")
corr.columns = ["feature_x", "feature_y", "correlation"]

heatmap = (
    alt.Chart(corr)
    .mark_rect()
    .encode(
        alt.X("feature_x:N", title="Feature X"),
        alt.Y("feature_y:N", title="Feature Y"),
        color=alt.Color(
            "correlation:Q", scale=alt.Scale(domain=(-1, 1),scheme="purpleorange"),
            title="Correlation"
        ),
        tooltip=["feature_x", "feature_y", "correlation"]
    )
    .properties(width=300, height=300, title="Correlation Heatmap")
)

heatmap = heatmap.properties(
    title="Correlation Heatmap of Wine Chemical Features"
).configure_title(fontSize=15)
heatmap

alt.Chart(...)

# Classification Model

# Results

# Discussion

# References